In [ ]:
from typing import Dict, List, TypedDict, Any, cast
from langgraph.graph import StateGraph, END

from langchain_community.embeddings import SentenceTransformerEmbeddings

In [21]:
import dotenv

dotenv.load_dotenv()

True

In [22]:
from langfuse import get_client
 
langfuse = get_client()
 
# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

Langfuse client is authenticated and ready!


In [48]:
import numpy as np
import json
import sqlite3
import sqlite_vec
from kiwipiepy import Kiwi


DB_PATH = "/home/codeitDev/project/AI_7-team/DB/document.db"
MODEL_NAME = "jhgan/ko-sroberta-multitask"

embeddings = SentenceTransformerEmbeddings(model_name=MODEL_NAME)

kiwi = Kiwi()

class SearchState(TypedDict):
    # 검색용 쿼리
    query: str

    # 계층 정보 (필터링 용도)
    scopes: List[Dict]

    # 검색 결과
    scoped_dense_result: List[Dict]
    scoped_sparse_result: List[Dict]
    dense_result: List[Dict]
    sparse_result: List[Dict]

    # 최종 결과
    search_result: List[Dict]


def hierarchy_search(state: SearchState):
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')
    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT
                json_extract(h.metadata, '$.doc_id'),
                json_extract(h.metadata, '$.level'),
                json_extract(h.metadata, '$.title'),
                json_extract(p.metadata, '$.level'),
                json_extract(p.metadata, '$.title'),
                v.distance
            FROM hierarchy h
            LEFT JOIN hierarchy p
            ON json_extract(h.metadata, '$.parent_uid')
                = json_extract(p.metadata, '$.uid')
            JOIN hierarchy_vec v ON h.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 100
        """, (query_vec,))
        results = cursor.fetchall()

        # 성적의 합을 통해, 가장 좋은 문서 50개를 반환한다.
        scores = {}
        counts = {}
        for row in results:
            if counts.get(row[0], 0) == 3:
                continue
            else:
                counts[row[0]] = counts.get(row[0], 0) + 1
                score = 1.0 / (1.0 + row[5])
                scores[row[0]] = scores.get(row[0], 0) + score
        
        sorted_docs = sorted(scores.items(), key=lambda x: -x[1])
        search_results = [doc_id for doc_id, score in sorted_docs[:30]]

    return {'scopes': search_results}


def extract_nouns(query):
    if not query:
        return ""
    tokens: List[Any] = cast(List[Any], kiwi.tokenize(query))
    nouns = [f'"{t.form}"' for t in tokens if t.tag in ('NNG', 'NNP', 'NNB')]
    if len(nouns) == 0:
        return ""
    return " OR ".join(nouns)


def scoped_dense_search(state: SearchState):
    scopes = state['scopes']
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')

    results = {}
    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        # 각 scope (계층 정보) 마다
        scope_query = f"""
            WITH scoped_chunks AS (
                SELECT *
                FROM chunks
                WHERE json_extract(metadata, '$.doc_id') IN (SELECT value FROM json_each(?))
            )
            SELECT
                json_extract(c.metadata, '$.uid'),
                c.text,
                c.metadata,
                v.distance
            FROM scoped_chunks c
            JOIN chunks_vec v ON c.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 30
        """
        cursor = conn.cursor()
        cursor.execute(scope_query, (json.dumps(list(scopes)), query_vec,))
        results = cursor.fetchall()
        results = [row for row in results]
        results = sorted(results, key=lambda x: x[3])
        
        # 'uid': ('text', 'metadata') 형식이다.
        results = [{result[0]:(result[1], json.loads(result[2]))} for result in results]

    return {'scoped_dense_result': results}


def scoped_sparse_search(state: SearchState):
    scopes = state['scopes']
    query = extract_nouns(state['query'].strip())
    # 단어 단위의 OR를 사용한다.
    # 추후 bigram 변경 계획이 있다.
    if len(query) == 0:
        return {'scoped_sparse_result': []}

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT 
                doc_id,
                uid,
                snippet(sparse, 1, '[', ']', '...', 20),
                bm25(sparse) as score
                FROM sparse
                WHERE nouns MATCH ? AND doc_id IN (SELECT value FROM json_each(?))
                ORDER BY score
                LIMIT 30
        """, (query, json.dumps(list(scopes)),))

        sparse_result = cursor.fetchall()
        uids = [r[1] for r in sparse_result]
        
        if len(uids) > 0:
            cursor.execute(f"""
                SELECT 
                    json_extract(metadata, '$.uid'), 
                    text,
                    metadata
                FROM chunks
                WHERE json_extract(metadata, '$.uid') IN (SELECT value FROM json_each(?))
            """, (json.dumps(list(uids)),))

            final_results = cursor.fetchall()

            final_results = {a: (b, json.loads(c)) for a, b, c in final_results}

            final_results = [{uid: final_results[uid]} for uid in uids]
        else:
            final_results = []
    if len(final_results) >= 30:
        final_results = final_results[:30]
    return {'scoped_sparse_result': final_results}


def dense_search(state: SearchState):
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT
                json_extract(c.metadata, '$.uid'),
                c.text,
                c.metadata
            FROM chunks c
            JOIN chunks_vec v ON c.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 30
        """, (query_vec,))
        results = cursor.fetchall()

        # 'uid': ('text', 'metadata') 형식이다.
        search_results = []
        for result in results:
            search_results.append({result[0]: (result[1], json.loads(result[2]))})

    return {'dense_result': search_results}


def sparse_search(state: SearchState):
    query = extract_nouns(state['query'].strip())
    # 단어 단위의 OR를 사용한다.
    # 추후 bigram 변경 계획이 있다.
    if len(query) == 0:
        return {'sparse_result': []}

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT 
                uid,
                snippet(sparse, 1, '[', ']', '...', 20),
                bm25(sparse) as score
                FROM sparse
                WHERE nouns MATCH ?
                ORDER BY score
                LIMIT 30
        """, (query,))

        sparse_result = cursor.fetchall()
        uids = [r[0] for r in sparse_result]
        
        if len(uids) > 0:
            cursor.execute(f"""
                SELECT 
                    json_extract(metadata, '$.uid'), 
                    text,
                    metadata
                FROM chunks
                WHERE json_extract(metadata, '$.uid') IN (SELECT value FROM json_each(?))
            """, (json.dumps(list(uids)),))

            final_results = cursor.fetchall()

            final_results = {a: (b, json.loads(c)) for a, b, c in final_results}

            final_results = [{uid: final_results[uid]} for uid in uids]
        else:
            final_results = []
    return {'sparse_result': final_results}


def rrf(state: SearchState):
    # RRF_K = 60
    RRF_K = 20
    
    def compute_scores(result_list):
        scores = {}
        for i, d in enumerate(result_list):
            doc_id = next(iter(d.keys()))
            scores[doc_id] = 1 / (i + RRF_K + 1)
        return scores

    # RRF 계산
    scoped_dense_scores = compute_scores(state['scoped_dense_result'])
    scoped_sparse_scores = compute_scores(state['scoped_sparse_result'])
    dense_scores = compute_scores(state['dense_result'])
    sparse_scores = compute_scores(state['sparse_result'],)

    # 검색된 전체 데이터에 대해, doc_id: title 매핑을 수행
    total_docs = {}
    for result_list in [
        state['scoped_dense_result'],
        state['scoped_sparse_result'],
        state['dense_result'],
        state['sparse_result']
    ]:
        for d in result_list:
            total_docs.update(d)

    # doc_id: score 매핑을 수행 - 동일 문서가 여러 번 나오면 합을 계산
    total_scores = {}

    for score_dict in [
        dense_scores,
        sparse_scores,
        scoped_dense_scores,
        scoped_sparse_scores
    ]:
        for doc_id, score in score_dict.items():
            total_scores[doc_id] = total_scores.get(doc_id, 0) + score

    score_list = sorted(total_scores.items(), key=lambda x: -x[1])

    result = [total_docs[k] for k, v in score_list[:30]]
    return {'search_result': result}

In [49]:
search_workflow = StateGraph(SearchState)

search_workflow.add_node("hierarchy", hierarchy_search)

search_workflow.add_node("scoped_dense", scoped_dense_search)
search_workflow.add_node("scoped_sparse", scoped_sparse_search)
search_workflow.add_node("dense", dense_search)
search_workflow.add_node("sparse", sparse_search)

search_workflow.add_node("rrf", rrf)

search_workflow.set_entry_point("hierarchy")

search_workflow.add_edge("hierarchy", "scoped_dense")
search_workflow.add_edge("hierarchy", "scoped_sparse")
search_workflow.add_edge("hierarchy", "dense")
search_workflow.add_edge("hierarchy", "sparse")

search_workflow.add_edge("scoped_dense", "rrf")
search_workflow.add_edge("scoped_sparse", "rrf")
search_workflow.add_edge("dense", "rrf")
search_workflow.add_edge("sparse", "rrf")
search_workflow.add_edge("rrf", END)

In [50]:
hybrid_app = search_workflow.compile()

In [46]:
class SearchState2(TypedDict):
    # 검색용 쿼리
    query: str

    # 계층 정보 (필터링 용도)
    scopes: List[Dict]

    # 검색 결과
    dense_result: List[Dict]

    # 최종 결과
    search_result: List[Dict]


def dense_search2(state: SearchState2):
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT
                json_extract(c.metadata, '$.uid'),
                c.text,
                c.metadata
            FROM chunks c
            JOIN chunks_vec v ON c.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 30
        """, (query_vec,))
        results = cursor.fetchall()

        # 'uid': ('text', 'metadata') 형식이다.
        search_results = []
        for result in results:
            search_results.append({result[0]: (result[1], json.loads(result[2]))})

    return {'dense_result': search_results}


def rrf2(state: SearchState2):
    # RRF_K = 60
    RRF_K = 20
    
    def compute_scores(result_list):
        scores = {}
        for i, d in enumerate(result_list):
            doc_id = next(iter(d.keys()))
            scores[doc_id] = 1 / (i + RRF_K + 1)
        return scores

    # RRF 계산
    dense_scores = compute_scores(state['dense_result'])

    # 검색된 전체 데이터에 대해, doc_id: title 매핑을 수행
    total_docs = {}
    for result_list in [
        state['dense_result']
    ]:
        for d in result_list:
            total_docs.update(d)

    # doc_id: score 매핑을 수행 - 동일 문서가 여러 번 나오면 합을 계산
    total_scores = {}

    for score_dict in [
        dense_scores,
    ]:
        for doc_id, score in score_dict.items():
            total_scores[doc_id] = total_scores.get(doc_id, 0) + score

    score_list = sorted(total_scores.items(), key=lambda x: -x[1])

    result = [total_docs[k] for k, v in score_list[:30]]
    return {'search_result': result}


search_workflow2 = StateGraph(SearchState2)

search_workflow2.add_node("dense", dense_search2)

search_workflow2.add_node("rrf", rrf2)

search_workflow2.set_entry_point("dense")
search_workflow2.add_edge("dense", "rrf")
search_workflow2.add_edge("rrf", END)

In [47]:
dense_app = search_workflow2.compile()

In [53]:
class SearchState3(TypedDict):
    # 검색용 쿼리
    query: str

    # 계층 정보 (필터링 용도)
    scopes: List[Dict]

    # 검색 결과
    scoped_dense_result: List[Dict]
    dense_result: List[Dict]

    # 최종 결과
    search_result: List[Dict]


def hierarchy_search3(state: SearchState3):
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')
    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT
                json_extract(h.metadata, '$.doc_id'),
                json_extract(h.metadata, '$.level'),
                json_extract(h.metadata, '$.title'),
                json_extract(p.metadata, '$.level'),
                json_extract(p.metadata, '$.title'),
                v.distance
            FROM hierarchy h
            LEFT JOIN hierarchy p
            ON json_extract(h.metadata, '$.parent_uid')
                = json_extract(p.metadata, '$.uid')
            JOIN hierarchy_vec v ON h.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 100
        """, (query_vec,))
        results = cursor.fetchall()

        # 성적의 합을 통해, 가장 좋은 문서 50개를 반환한다.
        scores = {}
        counts = {}
        for row in results:
            if counts.get(row[0], 0) == 3:
                continue
            else:
                counts[row[0]] = counts.get(row[0], 0) + 1
                score = 1.0 / (1.0 + row[5])
                scores[row[0]] = scores.get(row[0], 0) + score
        
        sorted_docs = sorted(scores.items(), key=lambda x: -x[1])
        search_results = [doc_id for doc_id, score in sorted_docs[:30]]

    return {'scopes': search_results}


def scoped_dense_search3(state: SearchState3):
    scopes = state['scopes']
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')

    results = {}
    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        # 각 scope (계층 정보) 마다
        scope_query = f"""
            WITH scoped_chunks AS (
                SELECT *
                FROM chunks
                WHERE json_extract(metadata, '$.doc_id') IN (SELECT value FROM json_each(?))
            )
            SELECT
                json_extract(c.metadata, '$.uid'),
                c.text,
                c.metadata,
                v.distance
            FROM scoped_chunks c
            JOIN chunks_vec v ON c.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 30
        """
        cursor = conn.cursor()
        cursor.execute(scope_query, (json.dumps(list(scopes)), query_vec,))
        results = cursor.fetchall()
        results = [row for row in results]
        results = sorted(results, key=lambda x: x[3])
        
        # 'uid': ('text', 'metadata') 형식이다.
        results = [{result[0]:(result[1], json.loads(result[2]))} for result in results]

    return {'scoped_dense_result': results}


def dense_search3(state: SearchState3):
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT
                json_extract(c.metadata, '$.uid'),
                c.text,
                c.metadata
            FROM chunks c
            JOIN chunks_vec v ON c.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 30
        """, (query_vec,))
        results = cursor.fetchall()

        # 'uid': ('text', 'metadata') 형식이다.
        search_results = []
        for result in results:
            search_results.append({result[0]: (result[1], json.loads(result[2]))})

    return {'dense_result': search_results}


def rrf3(state: SearchState3):
    # RRF_K = 60
    RRF_K = 20
    
    def compute_scores(result_list):
        scores = {}
        for i, d in enumerate(result_list):
            doc_id = next(iter(d.keys()))
            scores[doc_id] = 1 / (i + RRF_K + 1)
        return scores

    # RRF 계산
    scoped_dense_scores = compute_scores(state['scoped_dense_result'])
    dense_scores = compute_scores(state['dense_result'])

    # 검색된 전체 데이터에 대해, doc_id: title 매핑을 수행
    total_docs = {}
    for result_list in [
        state['scoped_dense_result'],
        state['dense_result'],
    ]:
        for d in result_list:
            total_docs.update(d)

    # doc_id: score 매핑을 수행 - 동일 문서가 여러 번 나오면 합을 계산
    total_scores = {}

    for score_dict in [
        dense_scores,
        scoped_dense_scores,
    ]:
        for doc_id, score in score_dict.items():
            total_scores[doc_id] = total_scores.get(doc_id, 0) + score

    score_list = sorted(total_scores.items(), key=lambda x: -x[1])

    result = [total_docs[k] for k, v in score_list[:30]]
    return {'search_result': result}

search_workflow3 = StateGraph(SearchState3)

search_workflow3.add_node("hierarchy", hierarchy_search3)

search_workflow3.add_node("scoped_dense", scoped_dense_search3)
search_workflow3.add_node("dense", dense_search3)

search_workflow3.add_node("rrf", rrf3)

search_workflow3.set_entry_point("hierarchy")

search_workflow3.add_edge("hierarchy", "scoped_dense")
search_workflow3.add_edge("hierarchy", "dense")

search_workflow3.add_edge("scoped_dense", "rrf")
search_workflow3.add_edge("dense", "rrf")
search_workflow3.add_edge("rrf", END)

In [54]:
dense_with_hierarchy_app = search_workflow3.compile()

In [58]:
import os
import re
import random


def split_paragraphs(md_text):
    # 빈 줄 기준 분리
    paragraphs = re.split(r"\n\s*\n", md_text)
    # 너무 짧은 문단 제거
    paragraphs = [p.strip() for p in paragraphs if len(p.strip()) > 200]
    return paragraphs


def split_sentences(paragraph):
    # 간단한 문장 분리 (과도한 regex 지양)
    sentences = re.split(r"(?<=[.!?다요])\s+", paragraph)
    return [s.strip() for s in sentences if len(s.strip()) > 30]


def sanitize_for_fts5(query: str) -> str:
    # 허용되는 문자만 남기고 제거
    return re.sub(r"[^가-힣a-zA-Z0-9\s]", "", query).strip()


def generate_queries_from_markdown(md_path):
    with open(md_path, "r", encoding="utf-8") as f:
        text = f.read()

    paragraphs = split_paragraphs(text)

    dense_queries = []
    sparse_queries = []
    hybrid_queries = []

    while len(dense_queries) < 2 or \
          len(sparse_queries) < 2 or \
          len(hybrid_queries) < 2:

        p = random.choice(paragraphs)
        sentences = split_sentences(p)

        if not sentences:
            continue

        s = random.choice(sentences)

        # ---- Dense ----
        if len(dense_queries) < 2:
            # q = s[start:end]
            q = s
            # if 60 <= len(q) <= 120:
            dense_queries.append({
                "query": q,
                "original": s
            })

        # ---- Sparse ----
        if len(sparse_queries) < 2:
            nouns = extract_nouns(s)
            if not nouns:
                continue
            nouns = nouns.strip().split()
            # if len(nouns) >= 5:
            # selected = nouns[:7]
            q = " ".join(nouns)
            q = sanitize_for_fts5(q)
            if len(q) >= 3:
                sparse_queries.append({
                    "query": q,
                    "original": s
                })

        # ---- Hybrid ----
        if len(hybrid_queries) < 2:
            nouns = extract_nouns(s)
            if not nouns:
                continue
            nouns = nouns.strip().split()
            # if len(nouns) >= 3:
            # selected = nouns[:3]
            
            '''start = random.randint(0, max(1, len(s)//3))
            end = min(len(s), start + 60)
            base = s[start:end]'''
            q = s + " " + " ".join(nouns)
            q = sanitize_for_fts5(q)
            if len(q) >= 3:
                hybrid_queries.append({
                    "query": q,
                    "original": s
                })

    return {
        "dense": dense_queries[:2],
        "sparse": sparse_queries[:2],
        "hybrid": hybrid_queries[:2],
    }

In [59]:
def generate_all_queries(folder_path):
    results = {}

    for file in os.listdir(folder_path):
        if file.endswith(".md") and file.startswith("step2"):
            path = os.path.join(folder_path, file)
            results[file] = generate_queries_from_markdown(path)

    return results

In [60]:
def run_self_retrieval_test(md_path, retriever):
    queries = generate_queries_from_markdown(md_path)

    stats = {
        "dense": {"hit": 0, "total": 0, "ranks": []},
        "sparse": {"hit": 0, "total": 0, "ranks": []},
        "hybrid": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["dense", "sparse", "hybrid"]:

        for item in queries[q_type]:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retriever.invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    break

            stats[q_type]["total"] += 1

    return stats

In [61]:
def run_full_evaluation(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "dense": {"hit": 0, "total": 0, "ranks": []},
        "sparse": {"hit": 0, "total": 0, "ranks": []},
        "hybrid": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])
        print(aggregate)

    return aggregate

In [62]:

def print_results(aggregate):

    total_hit = 0
    total_total = 0

    for k in ["dense", "sparse", "hybrid"]:
        hit = aggregate[k]["hit"]
        total = aggregate[k]["total"]
        ranks = aggregate[k]["ranks"]

        recall = hit / total if total else 0
        avg_rank = sum(ranks)/len(ranks) if ranks else None
        mrr = sum(1/r for r in ranks)/total if total else 0

        print(f"\n[{k.upper()}]")
        print(f"Recall: {recall:.3f}")
        print(f"Avg Rank: {avg_rank}")
        print(f"MRR: {mrr:.3f}")

        total_hit += hit
        total_total += total

    if total_total:
        print("\n[OVERALL]")
        print(f"Recall: {total_hit/total_total:.3f}")

In [63]:
def generate_queries_from_markdown2(md_path):
    with open(md_path, "r", encoding="utf-8") as f:
        text = f.read()

    paragraphs = split_paragraphs(text)

    queries = []

    while len(queries) < 2:

        p = random.choice(paragraphs)
        sentences = split_sentences(p)

        if not sentences:
            continue

        s = random.choice(sentences)

        # ---- Dense ----
        if len(queries) < 2:
            # q = s[start:end]
            q = s
            # if 60 <= len(q) <= 120:
            queries.append({
                "query": q,
                "original": s
            })

    return queries[:2]

In [64]:
def generate_all_queries2(folder_path):
    results = {}

    for file in os.listdir(folder_path):
        if file.endswith(".md") and file.startswith("step2"):
            path = os.path.join(folder_path, file)
            results[file] = generate_queries_from_markdown2(path)

    return results

In [65]:
def run_self_retrieval_test2(md_path, retrievers):
    queries = generate_queries_from_markdown2(md_path)

    stats = {
        "dense": {"hit": 0, "total": 0, "ranks": []},
        "dense_with_hierarchy": {"hit": 0, "total": 0, "ranks": []},
        "hybrid": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["dense", "dense_with_hierarchy", "hybrid"]:

        for item in queries:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retrievers[q_type].invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            hit = False

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    hit = True
                    break

            stats[q_type]["total"] += 1

    return stats

In [66]:
def run_full_evaluation2(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "dense": {"hit": 0, "total": 0, "ranks": []},
        "dense_with_hierarchy": {"hit": 0, "total": 0, "ranks": []},
        "hybrid": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test2(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])

    return aggregate

In [67]:
def print_results2(aggregate):

    total_hit = 0
    total_total = 0

    for k in ["dense", "dense_with_hierarchy", "hybrid"]:
        hit = aggregate[k]["hit"]
        total = aggregate[k]["total"]
        ranks = aggregate[k]["ranks"]

        recall = hit / total if total else 0
        avg_rank = sum(ranks)/len(ranks) if ranks else None
        mrr = sum(1/r for r in ranks)/total if total else 0

        print(f"\n[{k.upper()}]")
        print(f"Recall: {recall:.3f}")
        print(f"Avg Rank: {avg_rank}")
        print(f"MRR: {mrr:.3f}")

        total_hit += hit
        total_total += total

    if total_total:
        print("\n[OVERALL]")
        print(f"Recall: {total_hit/total_total:.3f}")

In [68]:
aggregate = run_full_evaluation2('/home/codeitDev/project/AI_7-team/output', {"dense": dense_app, "dense_with_hierarchy": dense_with_hierarchy_app, "hybrid": hybrid_app})
print_results2(aggregate)


[DENSE]
Recall: 0.585
Avg Rank: 2.0085470085470085
MRR: 0.508

[DENSE_WITH_HIERARCHY]
Recall: 0.585
Avg Rank: 6.743589743589744
MRR: 0.276

[HYBRID]
Recall: 0.860
Avg Rank: 6.308139534883721
MRR: 0.497

[OVERALL]
Recall: 0.677


In [110]:
class SearchState4(TypedDict):
    # 검색용 쿼리
    query: str

    dense_result: List[Dict]
    sparse_result: List[Dict]

    # 최종 결과
    search_result: List[Dict]


def dense_search4(state: SearchState4):
    query = state['query']
    query_vec = np.array(embeddings.embed_query(query)).astype('float32')

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT
                json_extract(c.metadata, '$.uid'),
                c.text,
                c.metadata
            FROM chunks c
            JOIN chunks_vec v ON c.rowid = v.rowid
            WHERE v.text_embedding MATCH ? AND k = 30
        """, (query_vec,))
        results = cursor.fetchall()

        # 'uid': ('text', 'metadata') 형식이다.
        search_results = []
        for result in results:
            search_results.append({result[0]: (result[1], json.loads(result[2]))})

    return {'dense_result': search_results}


def sparse_search4(state: SearchState4):
    query = extract_nouns(state['query'].strip())
    # 단어 단위의 OR를 사용한다.
    # 추후 bigram 변경 계획이 있다.
    if len(query) == 0:
        return {'sparse_result': []}

    with sqlite3.connect(DB_PATH) as conn:
        conn.enable_load_extension(True)
        sqlite_vec.load(conn)
        conn.enable_load_extension(False)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT 
                uid,
                snippet(sparse, 1, '[', ']', '...', 20),
                bm25(sparse) as score
                FROM sparse
                WHERE nouns MATCH ?
                ORDER BY score
                LIMIT 30
        """, (query,))

        sparse_result = cursor.fetchall()
        uids = [r[0] for r in sparse_result]
        
        if len(uids) > 0:
            cursor.execute(f"""
                SELECT 
                    json_extract(metadata, '$.uid'), 
                    text,
                    metadata
                FROM chunks
                WHERE json_extract(metadata, '$.uid') IN (SELECT value FROM json_each(?))
            """, (json.dumps(list(uids)),))

            final_results = cursor.fetchall()

            final_results = {a: (b, json.loads(c)) for a, b, c in final_results}

            final_results = [{uid: final_results[uid]} for uid in uids]
        else:
            final_results = []
    return {'sparse_result': final_results}


def rrf4(state: SearchState4):
    # RRF_K = 60
    RRF_K = 20
    
    def compute_scores(result_list):
        scores = {}
        for i, d in enumerate(result_list):
            doc_id = next(iter(d.keys()))
            scores[doc_id] = 1 / (i + RRF_K + 1)
        return scores

    # RRF 계산
    dense_scores = compute_scores(state['dense_result'])
    sparse_scores = compute_scores(state['sparse_result'],)

    # 검색된 전체 데이터에 대해, doc_id: title 매핑을 수행
    total_docs = {}
    for result_list in [
        state['dense_result'],
        state['sparse_result']
    ]:
        for d in result_list:
            total_docs.update(d)

    # doc_id: score 매핑을 수행 - 동일 문서가 여러 번 나오면 합을 계산
    total_scores = {}

    for score_dict in [
        dense_scores,
        sparse_scores
    ]:
        for doc_id, score in score_dict.items():
            total_scores[doc_id] = total_scores.get(doc_id, 0) + score

    score_list = sorted(total_scores.items(), key=lambda x: -x[1])

    result = [total_docs[k] for k, v in score_list[:30]]
    return {'search_result': result}

In [77]:
def empty4(state: SearchState4):
    return

In [78]:
search_workflow4 = StateGraph(SearchState4)

search_workflow4.add_node("empty", empty4)

search_workflow4.add_node("dense", dense_search4)
search_workflow4.add_node("sparse", sparse_search4)

search_workflow4.add_node("rrf", rrf4)

search_workflow4.set_entry_point("empty")

search_workflow4.add_edge("empty", "dense")
search_workflow4.add_edge("empty", "sparse")

search_workflow4.add_edge("dense", "rrf")
search_workflow4.add_edge("sparse", "rrf")
search_workflow4.add_edge("rrf", END)

In [79]:
hybrid_app2 = search_workflow4.compile()

In [84]:
def run_self_retrieval_test4(md_path, retrievers):
    queries = generate_queries_from_markdown2(md_path)

    stats = {
        "hybrid2": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["hybrid2"]:

        for item in queries:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retrievers[q_type].invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    break

            stats[q_type]["total"] += 1

    return stats

In [85]:
def run_full_evaluation4(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "hybrid2": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test4(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])

    return aggregate

In [86]:
def print_results4(aggregate):

    total_hit = 0
    total_total = 0

    for k in ["hybrid2"]:
        hit = aggregate[k]["hit"]
        total = aggregate[k]["total"]
        ranks = aggregate[k]["ranks"]

        recall = hit / total if total else 0
        avg_rank = sum(ranks)/len(ranks) if ranks else None
        mrr = sum(1/r for r in ranks)/total if total else 0

        print(f"\n[{k.upper()}]")
        print(f"Recall: {recall:.3f}")
        print(f"Avg Rank: {avg_rank}")
        print(f"MRR: {mrr:.3f}")

        total_hit += hit
        total_total += total

    if total_total:
        print("\n[OVERALL]")
        print(f"Recall: {total_hit/total_total:.3f}")

In [ ]:
aggregate = run_full_evaluation4('/home/codeitDev/project/AI_7-team/output', {"hybrid2": hybrid_app2})
print_results4(aggregate)


[HYBRID2]
Recall: 0.890
Avg Rank: 2.8202247191011236
MRR: 0.670

[OVERALL]
Recall: 0.890


In [88]:
def rrf5(state: SearchState4):
    # RRF_K = 60
    RRF_K = 20
    
    def compute_scores(result_list, w = 1):
        scores = {}
        for i, d in enumerate(result_list):
            doc_id = next(iter(d.keys()))
            scores[doc_id] = w / (i + RRF_K + 1)
        return scores

    # RRF 계산
    dense_scores = compute_scores(state['dense_result'], 3)
    sparse_scores = compute_scores(state['sparse_result'], 2)

    # 검색된 전체 데이터에 대해, doc_id: title 매핑을 수행
    total_docs = {}
    for result_list in [
        state['dense_result'],
        state['sparse_result']
    ]:
        for d in result_list:
            total_docs.update(d)

    # doc_id: score 매핑을 수행 - 동일 문서가 여러 번 나오면 합을 계산
    total_scores = {}

    for score_dict in [
        dense_scores,
        sparse_scores
    ]:
        for doc_id, score in score_dict.items():
            total_scores[doc_id] = total_scores.get(doc_id, 0) + score

    score_list = sorted(total_scores.items(), key=lambda x: -x[1])

    result = [total_docs[k] for k, v in score_list[:30]]
    return {'search_result': result}

In [89]:
search_workflow5 = StateGraph(SearchState4)

search_workflow5.add_node("empty", empty4)

search_workflow5.add_node("dense", dense_search4)
search_workflow5.add_node("sparse", sparse_search4)

search_workflow5.add_node("rrf", rrf5)

search_workflow5.set_entry_point("empty")

search_workflow5.add_edge("empty", "dense")
search_workflow5.add_edge("empty", "sparse")

search_workflow5.add_edge("dense", "rrf")
search_workflow5.add_edge("sparse", "rrf")
search_workflow5.add_edge("rrf", END)

In [90]:
hybrid_app3 = search_workflow5.compile()

In [94]:
def run_self_retrieval_test5(md_path, retrievers):
    queries = generate_queries_from_markdown2(md_path)

    stats = {
        "hybrid3": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["hybrid3"]:

        for item in queries:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retrievers[q_type].invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    break

            stats[q_type]["total"] += 1

    return stats

In [98]:
def run_full_evaluation5(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "hybrid3": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test5(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])

    return aggregate

In [99]:
def print_results5(aggregate):

    total_hit = 0
    total_total = 0

    for k in ["hybrid3"]:
        hit = aggregate[k]["hit"]
        total = aggregate[k]["total"]
        ranks = aggregate[k]["ranks"]

        recall = hit / total if total else 0
        avg_rank = sum(ranks)/len(ranks) if ranks else None
        mrr = sum(1/r for r in ranks)/total if total else 0

        print(f"\n[{k.upper()}]")
        print(f"Recall: {recall:.3f}")
        print(f"Avg Rank: {avg_rank}")
        print(f"MRR: {mrr:.3f}")

        total_hit += hit
        total_total += total

    if total_total:
        print("\n[OVERALL]")
        print(f"Recall: {total_hit/total_total:.3f}")

In [100]:
aggregate = run_full_evaluation5('/home/codeitDev/project/AI_7-team/output', {"hybrid3": hybrid_app3})
print_results5(aggregate)


[HYBRID3]
Recall: 0.870
Avg Rank: 5.672413793103448
MRR: 0.563

[OVERALL]
Recall: 0.870


In [130]:
def generate_queries_from_markdown6(md_path):
    with open(md_path, "r", encoding="utf-8") as f:
        text = f.read()

    paragraphs = split_paragraphs(text)

    queries = []

    while len(queries) < 10:

        p = random.choice(paragraphs)
        sentences = split_sentences(p)

        if not sentences:
            continue

        s = random.choice(sentences)

        # ---- Dense ----
        if len(queries) < 10:
            start = random.randint(0, max(1, len(s)//4))
            end = random.randint(len(s)//2, len(s))
            # q = s[start:end]
            q = s
            # if 60 <= len(q) <= 120:
            queries.append({
                "query": q,
                "original": s
            })

    return queries[:10]

In [131]:
def run_self_retrieval_test6(md_path, retrievers):
    queries = generate_queries_from_markdown6(md_path)

    stats = {
        "hybrid2": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["hybrid2"]:

        for item in queries:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retrievers[q_type].invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    break

            stats[q_type]["total"] += 1

    return stats

In [132]:
def run_full_evaluation6(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "hybrid2": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test6(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])

    return aggregate

In [133]:
for i in range(3):
    aggregate = run_full_evaluation6('/home/codeitDev/project/AI_7-team/output', {"hybrid2": hybrid_app2})
    print_results4(aggregate)


[HYBRID2]
Recall: 0.901
Avg Rank: 3.1542730299667037
MRR: 0.641

[OVERALL]
Recall: 0.901

[HYBRID2]
Recall: 0.896
Avg Rank: 3.193080357142857
MRR: 0.629

[OVERALL]
Recall: 0.896

[HYBRID2]
Recall: 0.894
Avg Rank: 3.1297539149888145
MRR: 0.642

[OVERALL]
Recall: 0.894


In [107]:
def run_self_retrieval_test7(md_path, retrievers):
    queries = generate_queries_from_markdown6(md_path)

    stats = {
        "hybrid3": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["hybrid3"]:

        for item in queries:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retrievers[q_type].invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    break

            stats[q_type]["total"] += 1

    return stats

In [108]:
def run_full_evaluation7(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "hybrid3": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test7(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])

    return aggregate

In [109]:
for i in range(3):
    aggregate = run_full_evaluation7('/home/codeitDev/project/AI_7-team/output', {"hybrid3": hybrid_app3})
    print_results5(aggregate)


[HYBRID3]
Recall: 0.891
Avg Rank: 5.418630751964085
MRR: 0.595

[OVERALL]
Recall: 0.891

[HYBRID3]
Recall: 0.883
Avg Rank: 5.731596828992073
MRR: 0.579

[OVERALL]
Recall: 0.883

[HYBRID3]
Recall: 0.874
Avg Rank: 5.384439359267734
MRR: 0.584

[OVERALL]
Recall: 0.874


In [111]:
def rrf6(state: SearchState4):
    # RRF_K = 60
    RRF_K = 20
    
    def compute_scores(result_list, w = 1):
        scores = {}
        for i, d in enumerate(result_list):
            doc_id = next(iter(d.keys()))
            scores[doc_id] = w / (i + RRF_K + 1)
        return scores

    # RRF 계산
    dense_scores = compute_scores(state['dense_result'], 2)
    sparse_scores = compute_scores(state['sparse_result'], 3)

    # 검색된 전체 데이터에 대해, doc_id: title 매핑을 수행
    total_docs = {}
    for result_list in [
        state['dense_result'],
        state['sparse_result']
    ]:
        for d in result_list:
            total_docs.update(d)

    # doc_id: score 매핑을 수행 - 동일 문서가 여러 번 나오면 합을 계산
    total_scores = {}

    for score_dict in [
        dense_scores,
        sparse_scores
    ]:
        for doc_id, score in score_dict.items():
            total_scores[doc_id] = total_scores.get(doc_id, 0) + score

    score_list = sorted(total_scores.items(), key=lambda x: -x[1])

    result = [total_docs[k] for k, v in score_list[:30]]
    return {'search_result': result}

In [112]:
search_workflow6 = StateGraph(SearchState4)

search_workflow6.add_node("empty", empty4)

search_workflow6.add_node("dense", dense_search4)
search_workflow6.add_node("sparse", sparse_search4)

search_workflow6.add_node("rrf", rrf6)

search_workflow6.set_entry_point("empty")

search_workflow6.add_edge("empty", "dense")
search_workflow6.add_edge("empty", "sparse")

search_workflow6.add_edge("dense", "rrf")
search_workflow6.add_edge("sparse", "rrf")
search_workflow6.add_edge("rrf", END)

In [113]:
hybrid_app4 = search_workflow6.compile()

In [126]:
def run_self_retrieval_test8(md_path, retrievers):
    queries = generate_queries_from_markdown6(md_path)

    stats = {
        "hybrid4": {"hit": 0, "total": 0, "ranks": []},
    }

    for q_type in ["hybrid4"]:

        for item in queries:
            query_text = item["query"]
            original_sentence = item["original"]

            # ✅ retriever 실행
            result = retrievers[q_type].invoke({"query": query_text})
            retrieved_chunks = result['search_result']  # 네 구조에 맞게 수정

            for rank, doc in enumerate(retrieved_chunks):
                if original_sentence.replace(" ", "").replace("\n", "") in doc[0].replace(" ", "").replace("\n", ""):
                    stats[q_type]["hit"] += 1
                    stats[q_type]["ranks"].append(rank + 1)
                    break

            stats[q_type]["total"] += 1

    return stats

In [127]:
def run_full_evaluation8(folder_path, retriever):
    files = [f for f in os.listdir(folder_path)
             if f.startswith("step2") and f.endswith(".md")]

    aggregate = {
        "hybrid4": {"hit": 0, "total": 0, "ranks": []},
    }

    for file in files:
        path = os.path.join(folder_path, file)
        stats = run_self_retrieval_test8(path, retriever)

        for k in aggregate:
            aggregate[k]["hit"] += stats[k]["hit"]
            aggregate[k]["total"] += stats[k]["total"]
            aggregate[k]["ranks"].extend(stats[k]["ranks"])

    return aggregate

In [128]:
def print_results6(aggregate):

    total_hit = 0
    total_total = 0

    for k in ["hybrid4"]:
        hit = aggregate[k]["hit"]
        total = aggregate[k]["total"]
        ranks = aggregate[k]["ranks"]

        recall = hit / total if total else 0
        avg_rank = sum(ranks)/len(ranks) if ranks else None
        mrr = sum(1/r for r in ranks)/total if total else 0

        print(f"\n[{k.upper()}]")
        print(f"Recall: {recall:.3f}")
        print(f"Avg Rank: {avg_rank}")
        print(f"MRR: {mrr:.3f}")

        total_hit += hit
        total_total += total

    if total_total:
        print("\n[OVERALL]")
        print(f"Recall: {total_hit/total_total:.3f}")

In [129]:
for i in range(3):
    aggregate = run_full_evaluation8('/home/codeitDev/project/AI_7-team/output', {"hybrid4": hybrid_app4})
    print_results6(aggregate)


[HYBRID4]
Recall: 0.896
Avg Rank: 2.622767857142857
MRR: 0.671

[OVERALL]
Recall: 0.896

[HYBRID4]
Recall: 0.926
Avg Rank: 2.5572354211663066
MRR: 0.712

[OVERALL]
Recall: 0.926

[HYBRID4]
Recall: 0.920
Avg Rank: 2.5706521739130435
MRR: 0.699

[OVERALL]
Recall: 0.920
